### Rede Neural Recorrente (RNN) para Previsão de Preços de Ações

Este exemplo demonstra como construir uma Rede Neural Recorrente para prever preços de ações. Usaremos uma arquitetura LSTM (Long Short-Term Memory).

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.preprocessing import MinMaxScaler
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense


In [ ]:
# Instalar a biblioteca yfinance para buscar dados reais
!pip install yfinance
import yfinance as yf

## 2 Buscar Dados Reais de Ações do Yahoo Finance

Vamos definir um ticker e um período para buscar os dados de preços de fechamento.

In [ ]:
# Definir o ticker da ação e o período
ticker_symbol = 'PETR4.SA' # Exemplo: Petrobras no Brasil
start_date = '2018-01-01'
end_date = '2023-01-01'

# Baixar dados históricos
data = yf.download(ticker_symbol, start=start_date, end=end_date)

# Usaremos a coluna 'Close' (Fechamento) para a previsão
data = data[['Close']]
data.rename(columns={'Close': 'Price'}, inplace=True)
display(data.head())
display(data.tail())

plt.figure(figsize=(12, 6))
plt.plot(data['Price'])
plt.title(f'Preços de Fechamento de {ticker_symbol}')
plt.xlabel('Data')
plt.ylabel('Preço')
plt.show()

### 3. Pré-processamento dos Dados

- **Normalização**: As RNNs geralmente funcionam melhor com dados escalonados (normalizados) entre 0 e 1.
- **Criação de Sequências**: Precisamos formatar os dados em sequências (janelas de tempo) para a entrada da RNN. Cada sequência será composta por `look_back` dias para prever o próximo dia.

In [ ]:
scaler = MinMaxScaler(feature_range=(0, 1))
scaled_data = scaler.fit_transform(data['Price'].values.reshape(-1, 1))

# Função para criar o dataset com janelas de tempo
def create_dataset(dataset, look_back=1):
    X, Y = [], []
    for i in range(len(dataset) - look_back):
        a = dataset[i:(i + look_back), 0]
        X.append(a)
        Y.append(dataset[i + look_back, 0])
    return np.array(X), np.array(Y)

look_back = 60 # Número de dias anteriores para considerar na previsão
X, y = create_dataset(scaled_data, look_back)

# Dividir em conjuntos de treino e teste
train_size = int(len(X) * 0.8)
X_train, X_test = X[0:train_size], X[train_size:len(X)]
y_train, y_test = y[0:train_size], y[train_size:len(y)]

# Redimensionar a entrada para o formato [amostras, janelas_de_tempo, features]
X_train = np.reshape(X_train, (X_train.shape[0], X_train.shape[1], 1))
X_test = np.reshape(X_test, (X_test.shape[0], X_test.shape[1], 1))

print(f"Formato de X_train: {X_train.shape}")
print(f"Formato de y_train: {y_train.shape}")

Formato de X_train: (943, 60, 1)
Formato de y_train: (943,)


### 4. Construir o Modelo RNN (LSTM)

Usaremos uma camada LSTM seguida por uma camada Dense para a saída.

In [ ]:
model = Sequential()
model.add(LSTM(units=50, return_sequences=True, input_shape=(look_back, 1)))
model.add(LSTM(units=50))
model.add(Dense(units=1))
model.compile(optimizer='adam', loss='mean_squared_error')
model.summary()

In [ ]:
model = Sequential()
model.add(LSTM(units=50, return_sequences=True, input_shape=(look_back, 1)))
model.add(LSTM(units=50))
model.add(Dense(units=1))

model.compile(optimizer='adam', loss='mean_squared_error')
model.summary()

### 5. Treinar o Modelo

In [ ]:
history = model.fit(X_train, y_train, epochs=20, batch_size=32, validation_split=0.1, verbose=1)

plt.figure(figsize=(10, 6))
plt.plot(history.history['loss'], label='Loss de Treino')
plt.plot(history.history['val_loss'], label='Loss de Validação')
plt.title('Histórico de Treinamento do Modelo')
plt.xlabel('Época')
plt.ylabel('Loss')
plt.legend()
plt.show()

### 6. Fazer Previsões e Avaliar o Modelo

In [ ]:
train_predict = model.predict(X_train)
test_predict = model.predict(X_test)

# Inverter a normalização para obter os preços originais
train_predict = scaler.inverse_transform(train_predict)
y_train = scaler.inverse_transform(y_train.reshape(-1, 1))
test_predict = scaler.inverse_transform(test_predict)
y_test = scaler.inverse_transform(y_test.reshape(-1, 1))

# Calcular métricas de avaliação
from sklearn.metrics import mean_squared_error
train_rmse = np.sqrt(mean_squared_error(y_train, train_predict))
test_rmse = np.sqrt(mean_squared_error(y_test, test_predict))
print(f'RMSE de Treino: {train_rmse:.2f}')
print(f'RMSE de Teste: {test_rmse:.2f}')

30/30 ━━━━━━━━━━━━━━━━━━━━ 1s 23ms/step
8/8 ━━━━━━━━━━━━━━━━━━━━ 0s 16ms/step
RMSE de Treino: 0.30
RMSE de Teste: 0.70


### 7. Visualizar Resultados

In [ ]:

plt.figure(figsize=(15, 7))
train_plot = np.empty_like(scaled_data)
train_plot[:, :] = np.nan
train_plot[look_back:len(train_predict) + look_back, :] = train_predict
test_plot = np.empty_like(scaled_data)
test_plot[:, :] = np.nan
# Correção do índice para plotar as previsões de teste corretamente
test_plot[train_size + look_back:len(scaled_data), :] = test_predict


plt.plot(data.index, scaler.inverse_transform(scaled_data), label='Preço Real')
plt.plot(data.index, train_plot, label='Previsão de Treino')
plt.plot(data.index, test_plot, label='Previsão de Teste')
plt.title('Previsão de Preços de Ações com RNN (LSTM)')
plt.xlabel('Data')
plt.ylabel('Preço')
plt.legend()
plt.show()